In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(r"C:\Users\patil\OneDrive\Documents\ai-hiring-bias-xai/data/processed/resumes_cleaned.csv")

print("Shape:", df.shape)
df.head()

Shape: (1500, 12)


,Resume_ID,Name,Skills,experience_years,Education,Certifications,Job Role,hired,salary_expectation,projects_count,ai_score,gender
0,1,Ashley Ali,"TensorFlow, NLP, Pytorch",10,B.Sc,Deep Learning Specialization,AI Researcher,1.0,104895,8,100,Male
1,2,Wesley Roman,"Deep Learning, Machine Learning, Python, SQL",10,MBA,Google ML,Data Scientist,1.0,113002,1,100,Female
2,3,Corey Sanchez,"Ethical Hacking, Cybersecurity, Linux",1,MBA,Deep Learning Specialization,Cybersecurity Analyst,1.0,71766,7,70,Male
3,4,Elizabeth Carney,"Python, Pytorch, TensorFlow",7,B.Tech,AWS Certified,AI Researcher,1.0,46848,0,95,Male
4,5,Julie Hill,"SQL, React, Java",4,PhD,Deep Learning Specialization,Software Engineer,1.0,87441,9,100,Male


In [3]:
df.columns

Index(['Resume_ID', 'Name', 'Skills', 'experience_years', 'Education',
       'Certifications', 'Job Role', 'hired', 'salary_expectation',
       'projects_count', 'ai_score', 'gender'],
      dtype='object')

In [4]:
X = df.drop(columns=["hired"])
y = df["hired"]

In [5]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical:", categorical_features)
print("Numeric:", numeric_features)

Categorical: ['Name', 'Skills', 'Education', 'Certifications', 'Job Role', 'gender']
Numeric: ['Resume_ID', 'experience_years', 'salary_expectation', 'projects_count', 'ai_score']


In [6]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [7]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

In [8]:
clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", model)
])

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
clf.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Resume_ID',
                                                   'experience_years',
                                                   'salary_expectation',
                                                   'projects_count',
                                                   'ai_score']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Name', 'Skills',
                                                   'Education',
                                                   'Certifications', 'Job Role',
                                                   'gender'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [11]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = clf.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.54
[[  0 100]
 [ 38 162]]
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       100
         1.0       0.62      0.81      0.70       200

    accuracy                           0.54       300
   macro avg       0.31      0.41      0.35       300
weighted avg       0.41      0.54      0.47       300



In [12]:
print("Baseline Model Observations:")
print("- Logistic Regression trained on cleaned data.")
print("- Automatic feature detection prevents schema mismatch.")
print("- This baseline will be audited for fairness and bias.")

Baseline Model Observations:
- Logistic Regression trained on cleaned data.
- Automatic feature detection prevents schema mismatch.
- This baseline will be audited for fairness and bias.
